# 09_gold_setup
Creates the Gold star schema (4 dimensions + 1 fact table), seeds the static dimensions, and creates the Gold reporting views. Safe to re-run.

**Star schema**
- `fact_game_daily` (grain: one row per game per load date on which the game was new or changed)
- `dim_game`, `dim_publisher`, `dim_date`, `dim_price_tier`

In [0]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "workspace", "steam"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

## 1. Dimension tables

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_date (
  date_key INT,
  full_date DATE,
  year INT,
  quarter INT,
  month INT,
  month_name STRING,
  day INT,
  day_of_week INT,
  day_name STRING,
  week_of_year INT,
  is_weekend BOOLEAN
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_price_tier (
  price_tier_key INT,
  tier_name STRING,
  min_price_usd DOUBLE,
  max_price_usd DOUBLE,
  sort_order INT
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_publisher (
  publisher_key BIGINT,
  publisher_name STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_game (
  game_key INT,
  name STRING,
  developer STRING,
  first_seen_date DATE,
  last_changed_date DATE,
  load_timestamp TIMESTAMP
) USING DELTA
""")

## 2. Fact table (Liquid Clustering on the two join keys)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS fact_game_daily (
  game_key INT,
  date_key INT,
  publisher_key BIGINT,
  price_tier_key INT,
  ccu INT,
  positive INT,
  negative INT,
  total_reviews INT,
  approval_rate DOUBLE,
  owners_min BIGINT,
  owners_max BIGINT,
  owners_mid DOUBLE,
  price_usd DOUBLE,
  initialprice_usd DOUBLE,
  discount_pct DOUBLE,
  row_hash STRING,
  load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key, game_key)
""")

## 3. Seed the static dimensions (only when empty)

In [0]:
# dim_date: every calendar day from 2026 to 2030
if spark.table("dim_date").count() == 0:
    (spark.sql("SELECT explode(sequence(DATE'2026-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS full_date")
        .select(
            F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
            "full_date",
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.dayofmonth("full_date").alias("day"),
            (F.expr("weekday(full_date)") + 1).alias("day_of_week"),      # Monday = 1 ... Sunday = 7
            F.date_format("full_date", "EEEE").alias("day_name"),
            F.weekofyear("full_date").alias("week_of_year"),
            (F.expr("weekday(full_date)") >= 5).alias("is_weekend"))
        .write.mode("append").saveAsTable("dim_date"))

# dim_price_tier: 0 = Unknown handles a missing price
if spark.table("dim_price_tier").count() == 0:
    tiers = [(0, "Unknown", None, None, 0),
             (1, "Free", 0.0, 0.0, 1),
             (2, "Under $5", 0.01, 4.99, 2),
             (3, "$5 to $20", 5.0, 20.0, 3),
             (4, "Above $20", 20.01, None, 4)]
    (spark.createDataFrame(tiers, "price_tier_key INT, tier_name STRING, min_price_usd DOUBLE, max_price_usd DOUBLE, sort_order INT")
        .write.mode("append").saveAsTable("dim_price_tier"))

# dim_publisher: key 0 = Unknown handles a missing publisher
spark.sql("""
    MERGE INTO dim_publisher t
    USING (SELECT CAST(0 AS BIGINT) AS publisher_key, 'Unknown' AS publisher_name) s
    ON t.publisher_key = s.publisher_key
    WHEN NOT MATCHED THEN INSERT (publisher_key, publisher_name, load_timestamp)
    VALUES (s.publisher_key, s.publisher_name, current_timestamp())
""")

for t in ["dim_date", "dim_price_tier", "dim_publisher", "dim_game", "fact_game_daily"]:
    print(f"{t}: {spark.table(t).count()} rows")

## 4. Gold reporting views (always current, nothing to refresh)

In [0]:
# latest known row per game (the fact only holds rows on days a game was new or changed)
spark.sql("""
CREATE OR REPLACE VIEW vw_fact_latest AS
SELECT game_key, date_key, publisher_key, price_tier_key, ccu, positive, negative, total_reviews,
       approval_rate, owners_min, owners_max, owners_mid, price_usd, initialprice_usd, discount_pct
FROM (SELECT f.*, ROW_NUMBER() OVER (PARTITION BY f.game_key ORDER BY f.date_key DESC) AS rn
      FROM fact_game_daily f)
WHERE rn = 1
""")

spark.sql("""
CREATE OR REPLACE VIEW gold_publisher_summary AS
SELECT p.publisher_key,
       p.publisher_name,
       COUNT(*)                         AS games,
       ROUND(AVG(f.approval_rate), 4)   AS avg_approval_rate,
       SUM(f.owners_mid)                AS total_est_owners,
       ROUND(AVG(f.price_usd), 2)       AS avg_price_usd,
       SUM(f.ccu)                       AS total_ccu
FROM vw_fact_latest f
JOIN dim_publisher p ON f.publisher_key = p.publisher_key
GROUP BY p.publisher_key, p.publisher_name
""")

spark.sql("""
CREATE OR REPLACE VIEW gold_top_games AS
SELECT g.game_key, g.name, g.developer, p.publisher_name, t.tier_name AS price_tier,
       f.ccu, f.price_usd, f.approval_rate, f.total_reviews, f.owners_min, f.owners_max
FROM vw_fact_latest f
JOIN dim_game g       ON f.game_key = g.game_key
JOIN dim_publisher p  ON f.publisher_key = p.publisher_key
JOIN dim_price_tier t ON f.price_tier_key = t.price_tier_key
ORDER BY f.ccu DESC
LIMIT 500
""")

spark.sql("""
CREATE OR REPLACE VIEW gold_price_tier_analysis AS
SELECT t.price_tier_key, t.tier_name, t.sort_order,
       COUNT(*)                       AS games,
       ROUND(AVG(f.approval_rate), 4) AS avg_approval_rate,
       ROUND(AVG(f.ccu), 1)           AS avg_ccu,
       SUM(f.ccu)                     AS total_ccu
FROM vw_fact_latest f
JOIN dim_price_tier t ON f.price_tier_key = t.price_tier_key
GROUP BY t.price_tier_key, t.tier_name, t.sort_order
""")

display(spark.sql("SHOW TABLES"))